Create the Behaviour Risk Engine

In [2]:
import pandas as pd
import numpy as np

1. Load the dataset

In [74]:
# Human-readable dataset
activities = pd.read_csv("DBE_Insider_Threat_Dataset.csv")

# Machine learning results
model_results = pd.read_csv("DBE_Model_Results.csv")

print(activities.shape)
print(model_results.shape)

C:\Users\Admin\AppData\Local\Temp\ipykernel_21228\4139763005.py:2: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  activities = pd.read_csv("DBE_Insider_Threat_Dataset.csv")


(50000, 29)
(50000, 33)


Copy the ML results into the readable dataset

In [77]:
# Copy the Machine Learning results

activities["iso_prediction"] = model_results["iso_prediction"]
activities["cluster"] = model_results["cluster"]
activities["kmeans_prediction"] = model_results["kmeans_prediction"]

# We will create a smarter final prediction later

In [79]:
print(activities.columns)

Index(['timestamp', 'user_id', 'location', 'action', 'filename', 'url',
       'to_email', 'from_email', 'role', 'department', 'province',
       'subject_permission', 'hour', 'day', 'month', 'weekday', 'after_hours',
       'weekend', 'exam_period', 'records_accessed', 'days_to_exam',
       'insider_threat', 'work_email', 'accessed_subject',
       'allowed_exam_access', 'handled_exam_paper', 'unauthorised_exam_access',
       'risk_score', 'risk_level', 'iso_prediction', 'cluster',
       'kmeans_prediction'],
      dtype='object')


2. Create the Behaviour Risk Score

In [82]:
activities["behaviour_score"] = 0

3. After-hours activity

In [85]:
activities.loc[
    activities["after_hours"] == 1,
    "behaviour_score"
] += 10

4. Weekend activity

In [88]:
activities.loc[
    activities["weekend"] == 1,
    "behaviour_score"
] += 5

5. During examination period

In [91]:
activities.loc[
    activities["exam_period"] == 1,
    "behaviour_score"
] += 15

6. High number of records accessed

In [94]:
activities.loc[
    activities["records_accessed"] >= 15,
    "behaviour_score"
] += 20

7. Exam paper access

In [97]:
activities.loc[
    activities["handled_exam_paper"] == True,
    "behaviour_score"
] += 20

8. Unauthorised paper access

In [100]:
activities.loc[
    activities["unauthorised_exam_access"] == True,
    "behaviour_score"
] += 35

9. Isolation Forest detected anomaly

In [103]:
activities.loc[
    activities["iso_prediction"] == 1,
    "behaviour_score"
] += 25

10. KMeans detected anomaly

In [106]:
activities.loc[
    activities["kmeans_prediction"] == 1,
    "behaviour_score"
] += 25

Create Threat Levels

In [109]:
activities["threat_level"] = "Low"

activities.loc[
    activities["behaviour_score"] >= 40,
    "threat_level"
] = "Medium"

activities.loc[
    activities["behaviour_score"] >= 70,
    "threat_level"
] = "High"

activities.loc[
    activities["behaviour_score"] >= 90,
    "threat_level"
] = "Critical"

Final Decision

In [112]:
activities["final_prediction"] = (
    (
        (activities["iso_prediction"] == 1)
        |
        (activities["kmeans_prediction"] == 1)
    )
    &
    (
        activities["behaviour_score"] >= 70
    )
).astype(int)

Alert Message

In [115]:
activities["alert_message"] = "No Alert"

*High Risk

In [118]:
activities.loc[
    activities["threat_level"] == "High",
    "alert_message"
] = "Investigate Immediately"

*Critical

In [121]:
activities.loc[
    activities["threat_level"] == "Critical",
    "alert_message"
] = "Immediate Security Response Required"

Recommendation

In [124]:
activities["recommendation"] = "Continue Monitoring"

*Medium

In [127]:
activities.loc[
    activities["threat_level"] == "Medium",
    "recommendation"
] = "Review Employee Activity"

*High

In [142]:
activities.loc[
    activities["threat_level"] == "High",
    "recommendation"
] = "Launch Internal Investigation"

*Critical

In [145]:
activities.loc[
    activities["threat_level"] == "Critical",
    "recommendation"
] = "Suspend Account and Notify DBE Security"

In [147]:
activities["risk_reason"] = ""

In [149]:
for i in activities.index:

    reasons = []

    if activities.at[i, "after_hours"] == 1:
        reasons.append("After-hours activity")

    if activities.at[i, "weekend"] == 1:
        reasons.append("Weekend activity")

    if activities.at[i, "exam_period"] == 1:
        reasons.append("Exam period access")

    if activities.at[i, "records_accessed"] >= 15:
        reasons.append("High record access")

    if activities.at[i, "unauthorised_exam_access"]:
        reasons.append("Unauthorised exam paper access")

    if activities.at[i, "iso_prediction"] == 1:
        reasons.append("Isolation Forest anomaly")

    if activities.at[i, "kmeans_prediction"] == 1:
        reasons.append("KMeans anomaly")

    if len(reasons) == 0:
        activities.at[i, "risk_reason"] = "Normal employee activity"
    else:
        activities.at[i, "risk_reason"] = ", ".join(reasons)

Save the dataset

In [152]:
activities.to_csv(
    "SIZOKUTHOLA_Final_Dataset.csv",
    index=False
)

print("SIZOKUTHOLA dataset created successfully.")

SIZOKUTHOLA dataset created successfully.


In [137]:
print(activities.shape)
activities.info()
activities.head()

(50000, 37)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 37 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   timestamp                 50000 non-null  object
 1   user_id                   50000 non-null  object
 2   location                  50000 non-null  object
 3   action                    50000 non-null  object
 4   filename                  10517 non-null  object
 5   url                       10000 non-null  object
 6   to_email                  10300 non-null  object
 7   from_email                10300 non-null  object
 8   role                      50000 non-null  object
 9   department                50000 non-null  object
 10  province                  50000 non-null  object
 11  subject_permission        50000 non-null  object
 12  hour                      50000 non-null  int64 
 13  day                       50000 non-null  int64 
 14  month     

,timestamp,user_id,location,action,filename,url,to_email,from_email,role,department,...,risk_score,risk_level,iso_prediction,cluster,kmeans_prediction,behaviour_score,threat_level,final_prediction,alert_message,recommendation
0,2010-01-02 06:49:00,NGF0157,PC-6056,Logon,NaN,NaN,NaN,NaN,Senior Examination Officer,Results Processing,...,30,Low,0,1,0,15,Low,0,No Alert,Continue Monitoring
1,2010-01-02 06:50:00,LRR0148,PC-4275,Logon,NaN,NaN,NaN,NaN,Data Capturer,Logistics,...,30,Low,0,1,0,15,Low,0,No Alert,Continue Monitoring
2,2010-01-02 06:53:04,LRR0148,PC-4124,Logon,NaN,NaN,NaN,NaN,Data Capturer,Logistics,...,30,Low,0,1,0,15,Low,0,No Alert,Continue Monitoring
3,2010-01-02 07:00:00,IRM0931,PC-7188,Logon,NaN,NaN,NaN,NaN,ICT Administrator,Examination Administration,...,10,Low,0,1,0,5,Low,0,No Alert,Continue Monitoring
4,2010-01-02 07:00:00,MOH0273,PC-6699,Logon,NaN,NaN,NaN,NaN,Warehouse Officer,Examination Administration,...,10,Low,0,1,0,5,Low,0,No Alert,Continue Monitoring


In [140]:
print(activities.shape)

activities[[
    "user_id",
    "role",
    "action",
    "filename",
    "work_email",
    "risk_score",
    "behaviour_score",
    "threat_level",
    "iso_prediction",
    "kmeans_prediction",
    "final_prediction"
]].head()

(50000, 37)


,user_id,role,action,filename,work_email,risk_score,behaviour_score,threat_level,iso_prediction,kmeans_prediction,final_prediction
0,NGF0157,Senior Examination Officer,Logon,NaN,ngf0157@dbe.gov.za,30,15,Low,0,0,0
1,LRR0148,Data Capturer,Logon,NaN,lrr0148@dbe.gov.za,30,15,Low,0,0,0
2,LRR0148,Data Capturer,Logon,NaN,lrr0148@dbe.gov.za,30,15,Low,0,0,0
3,IRM0931,ICT Administrator,Logon,NaN,irm0931@dbe.gov.za,10,5,Low,0,0,0
4,MOH0273,Warehouse Officer,Logon,NaN,moh0273@dbe.gov.za,10,5,Low,0,0,0
